# Paper 3 — H4a: Does FR2 Confine Clusters Once Diffraction Is Modeled?

M3 found FR1 and FR2 clusters statistically indistinguishable (median 4/4, singletons
17.5%/19.5%), but both bands had been traced with Sionna's `diffraction=False` default — and edge
diffraction is precisely the mechanism that fills NLoS shadows at 3.5 GHz far more than at 28 GHz.
This notebook re-traces the **first 1000 UEs of the shared outdoor set in BOTH bands with
`diffraction=True`** and compares the 2x2 (band x diffraction) cluster statistics; the
no-diffraction halves come from the existing M1 and M3 files at zero cost.

**Pre-registered criteria (fixed before tracing):**
1. *Mechanism check* — with diffraction ON, FR1 singletons must DECREASE vs FR1 without
   diffraction (shadows get filled). If they don't, diffraction is not doing physical work in this
   scene and the test is uninformative.
2. *Confinement (H4a)* — with diffraction ON in both bands, FR2 shows more confinement than FR1:
   singleton fraction higher by more than 3 pp, or median cluster size strictly smaller. If the two
   bands remain equal within noise even with diffraction, H4a is reframed: in a deployment this
   dense the propagation contrast does not translate into cluster contrast, and the value of FR2
   for cluster work rests on delay resolution (the M3 sub-meter result), not on coverage.

Cost note: diffraction multiplies tracing time; energies only are needed, so no CFR is computed.
Runtime: GPU T4.

In [ ]:
# %% [1] Environment setup — safe to re-run; auto-restarts ONCE after installing
import importlib.util, subprocess, sys

if importlib.util.find_spec('sionna') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'sionna==2.0.1', 'h5py'], check=True)
    print('Install done — the runtime will now restart (EXPECTED).')
    print('When it comes back, run this cell again: it will skip the install.')
    import os, time; time.sleep(1); os.kill(os.getpid(), 9)

!nvidia-smi -L
import sionna, sionna.rt
print('Sionna:', sionna.__version__, '| Sionna RT:', sionna.rt.__version__)

In [ ]:
# %% [2] Drive mount + canary
import os, uuid, glob, time
import numpy as np
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

BASE = '/content/drive/MyDrive/paper3'
M1_DIR, M2_DIR, M3_DIR = (os.path.join(BASE, d) for d in ('m1', 'm2', 'm3'))
H4_DIR = os.path.join(BASE, 'h4a'); os.makedirs(H4_DIR, exist_ok=True)

def drive_canary(path=H4_DIR):
    token = str(uuid.uuid4()); c = os.path.join(path, '.canary')
    with open(c, 'w') as f: f.write(token)
    with open(c) as f: assert f.read() == token, 'DRIVE CANARY FAILED: phantom mount.'
    os.remove(c); print('Drive canary OK:', path)

drive_canary()

In [ ]:
# %% [3] Configuration (geometry identical to M1/M3; two band profiles)
N_POINTS = 1000
MAX_DEPTH = 5
GAMMA_DB = -10.0
SEED = 2026
BANDS = {
    'fr1': dict(carrier=3.5e9, tx_rows=2, tx_cols=4, batch=48),
    'fr2': dict(carrier=28e9, tx_rows=8, tx_cols=8, batch=24),
}
AREA_CENTER, UE_H, TX_H = (0.0, 50.0), 1.5, 7.0

ue_file = os.path.join(M1_DIR, 'ue_positions_A12_N5000_outdoor_seed2026.npy')
UE_POS = np.load(ue_file)[:N_POINTS]
tx_file = os.path.join(M2_DIR, 'tx_positions_A12.npy')
TX_POS = np.load(tx_file)
A = TX_POS.shape[0]
print(f'{N_POINTS} UEs (shared outdoor set) | {A} arrays | diffraction=True, max_depth={MAX_DEPTH}')

In [ ]:
# %% [4] Trace both bands with diffraction=True (energies + first-path ToA only, no CFR)
import mitsuba as mi
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, PathSolver

def trace_band(tag, cfg):
    out_npz = os.path.join(H4_DIR, f'h4a_{tag}_diffracted_N{N_POINTS}.npz')
    if os.path.exists(out_npz):
        print(f'{tag}: already traced, skipping'); return
    scene = load_scene(sionna.rt.scene.munich, merge_shapes=True)
    scene.frequency = cfg['carrier']
    scene.tx_array = PlanarArray(num_rows=cfg['tx_rows'], num_cols=cfg['tx_cols'],
                                 vertical_spacing=0.5, horizontal_spacing=0.5,
                                 pattern='tr38901', polarization='V')
    scene.rx_array = PlanarArray(num_rows=1, num_cols=1, vertical_spacing=0.5,
                                 horizontal_spacing=0.5, pattern='iso', polarization='V')
    for a, p in enumerate(TX_POS):
        tx = Transmitter(name=f'tx_{a}', position=[float(v) for v in p])
        scene.add(tx)
        tx.look_at([AREA_CENTER[0], AREA_CENTER[1], UE_H])
    solver = PathSolver()
    E_all, toa_all = [], []
    n_b = int(np.ceil(N_POINTS/cfg['batch']))
    t0 = time.time()
    for b in range(n_b):
        sl = slice(b*cfg['batch'], min((b+1)*cfg['batch'], N_POINTS))
        names = []
        for i, p in enumerate(UE_POS[sl]):
            scene.add(Receiver(name=f'rx_{i}', position=[float(v) for v in p]))
            names.append(f'rx_{i}')
        paths = solver(scene, max_depth=MAX_DEPTH, diffraction=True)
        a_cir, tau = (np.asarray(x) for x in paths.cir(normalize_delays=False, out_type='numpy'))
        for n in names: scene.remove(n)
        p_path = (np.abs(a_cir[..., 0])**2).sum(axis=(1, 3))
        tau_l = tau if tau.ndim == 3 else tau[:, 0, :, 0, :]
        valid = tau_l >= 0
        E_all.append(10*np.log10(np.where(valid, p_path, 0).sum(-1) + 1e-30))
        toa_all.append(np.where(valid, tau_l, np.inf).min(-1))
        el = time.time()-t0
        print(f'{tag} batch {b+1}/{n_b} | {el:5.1f} s | ETA {el/(b+1)*(n_b-b-1):5.0f} s '
              f'| {el/(b+1):4.1f} s/batch')
    drive_canary()
    np.savez_compressed(out_npz, E=np.concatenate(E_all).astype(np.float32),
                        toa=np.concatenate(toa_all).astype(np.float32))
    print(f'{tag}: saved {out_npz}')

for tag, cfg in BANDS.items():
    trace_band(tag, cfg)

In [ ]:
# %% [5] 2x2 comparison: band x diffraction on the SAME 1000 UEs
import h5py

def cluster_stats(E, label):
    mem = E >= E.max(1, keepdims=True) + GAMMA_DB
    sz = mem.sum(1)
    print(f'{label:28s} median {np.median(sz):2.0f} | mean {sz.mean():4.2f} | '
          f'singletons {np.mean(sz==1)*100:5.1f}% | size>=6 {np.mean(sz>=6)*100:5.1f}%')
    return sz, np.mean(sz == 1)

# no-diffraction halves from the existing files, restricted to the first-1000 subset by position
key = {tuple(np.round(p, 3)): i for i, p in enumerate(UE_POS)}

with h5py.File(os.path.join(M1_DIR, 'paper3_static_fr1_A12_N5000_outdoor_seed2026.h5'), 'r') as f:
    pos1, E1 = f['positions'][:], f['reports/E_dB_q'][:]
m1 = np.array([key.get(tuple(np.round(p, 3)), -1) for p in pos1])
E_fr1_nod = E1[m1 >= 0]

m3_h5 = sorted(glob.glob(os.path.join(M3_DIR, 'paper3_M3_fr2_*_paired.h5')))[-1]
with h5py.File(m3_h5, 'r') as f:
    pos3, E3 = f['positions'][:], f['fr2/E_dB_q'][:]
m3 = np.array([key.get(tuple(np.round(p, 3)), -1) for p in pos3])
E_fr2_nod = E3[m3 >= 0]

E_fr1_dif = np.load(os.path.join(H4_DIR, f'h4a_fr1_diffracted_N{N_POINTS}.npz'))['E']
E_fr2_dif = np.load(os.path.join(H4_DIR, f'h4a_fr2_diffracted_N{N_POINTS}.npz'))['E']

print(f'(no-diffraction subsets: FR1 {len(E_fr1_nod)}, FR2 {len(E_fr2_nod)} of {N_POINTS})')
_, s1n = cluster_stats(E_fr1_nod, 'FR1  no diffraction')
_, s1d = cluster_stats(E_fr1_dif, 'FR1  WITH diffraction')
_, s2n = cluster_stats(E_fr2_nod, 'FR2  no diffraction')
_, s2d = cluster_stats(E_fr2_dif, 'FR2  WITH diffraction')

sz1d = (E_fr1_dif >= E_fr1_dif.max(1, keepdims=True) + GAMMA_DB).sum(1)
sz2d = (E_fr2_dif >= E_fr2_dif.max(1, keepdims=True) + GAMMA_DB).sum(1)
n = len(sz1d)
se = np.sqrt(max(s1d*(1-s1d), s2d*(1-s2d))/n)
print()
print(f'[CHECK 1] diffraction fills FR1 shadows (FR1 singletons drop): '
      f'{"PASS" if s1d < s1n else "FAIL"} ({s1n*100:.1f}% -> {s1d*100:.1f}%)')
conf = (s2d - s1d > 0.03) or (np.median(sz2d) < np.median(sz1d))
print(f'[CHECK 2] H4a confinement with diffraction ON (FR2 singletons - FR1 > 3 pp, '
      f'or median strictly smaller): {"SUPPORTED" if conf else "NOT SUPPORTED"} '
      f'(singleton gap {100*(s2d-s1d):+.1f} pp, SE ~{se*100:.1f} pp; '
      f'medians {np.median(sz2d):.0f} vs {np.median(sz1d):.0f})')
print()
print('If NOT SUPPORTED: in this 12-array deployment the band contrast does not translate into')
print('cluster contrast, and the cluster-work value of FR2 rests on delay resolution (M3), not')
print('coverage confinement — reframe H4a accordingly in the paper.')